# Notebook 2: Advanced Retrieval Techniques

**Duration:** 120 minutes  
**Level:** Intermediate to Advanced  

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Implement semantic search using dense embeddings
- ✅ Build hybrid retrieval systems (BM25 + embeddings)
- ✅ Apply reranking for improved precision
- ✅ Implement query expansion techniques
- ✅ Evaluate retrieval systems with comprehensive metrics
- ✅ Optimize retrieval strategies for your use case

---

## Table of Contents

1. [Introduction: Beyond Keyword Search](#1-introduction-beyond-keyword-search)
2. [Environment Setup](#2-environment-setup)
3. [Dense Embeddings & Semantic Search](#3-dense-embeddings-semantic-search)
4. [Hybrid Retrieval: Best of Both Worlds](#4-hybrid-retrieval-best-of-both-worlds)
5. [Query Expansion Techniques](#5-query-expansion-techniques)
6. [Reranking Strategies](#6-reranking-strategies)
7. [Comprehensive Evaluation](#7-comprehensive-evaluation)
8. [Performance Optimization](#8-performance-optimization)
9. [Choosing the Right Strategy](#9-choosing-the-right-strategy)
10. [Exercises & Challenges](#10-exercises-challenges)
11. [Summary & Next Steps](#11-summary-next-steps)

---

## 1. Introduction: Beyond Keyword Search

### Limitations of BM25 (Keyword Search)

In Notebook 1, we used BM25 for retrieval. While effective, it has limitations:

| Limitation | Example |
|------------|----------|
| **Exact matching required** | Query: "car" won't match "automobile" |
| **No semantic understanding** | "How to cook pasta" vs "pasta cooking instructions" |
| **Vocabulary mismatch** | Medical: "myocardial infarction" vs "heart attack" |
| **Context ignored** | "bank" (financial) vs "bank" (river) |

### Dense Retrieval (Embeddings)

Dense retrieval uses neural networks to create **semantic vectors**:
- Similar meanings = similar vectors
- Captures context and relationships
- Language and vocabulary independent

### Hybrid Retrieval

Combines both approaches:
- **BM25**: Precise keyword matching
- **Dense**: Semantic similarity
- **Hybrid**: Best of both worlds

### Retrieval Pipeline Evolution

```
Basic:    Query → BM25 → Top K Docs → LLM

Advanced: Query → Embedder → Dense Retrieval ↘
                → BM25 Retrieval          → Join → Rerank → Top K → LLM
```

## 2. Environment Setup

In [ ]:
#CELL-NO: 1
# Standard library imports
import os
import json
import time
import warnings
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack core imports
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.utils import Secret

# Haystack retrieval imports
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever, InMemoryEmbeddingRetriever
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder
)

# Haystack joining and ranking
from haystack.components.joiners import DocumentJoiner
from haystack.components.rankers import (
    TransformersSimilarityRanker,
    LostInTheMiddleRanker
)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All imports successful!")

In [ ]:
#CELL-NO: 2
# Load environment variables
load_dotenv()

# Verify API keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")  # For reranking

if not OPENAI_API_KEY:
    raise ValueError("⚠️ OPENAI_API_KEY not found in .env file")

print("✅ Environment configured")
print(f"📍 OpenAI API Key: {OPENAI_API_KEY[:20]}...")
if COHERE_API_KEY:
    print(f"📍 Cohere API Key: {COHERE_API_KEY[:20]}...")
else:
    print("⚠️  Cohere API Key not found (optional for this notebook)")

## 3. Dense Embeddings & Semantic Search

Let's implement semantic search using sentence transformers.

### 3.1 Understanding Embeddings

Embeddings convert text into dense vectors (typically 384-1024 dimensions).

In [ ]:
#CELL-NO: 3
# Initialize a simple text embedder
text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
text_embedder.warm_up()

# Create embeddings for sample texts
texts = [
    "The cat sat on the mat",
    "A feline rested on the rug",  # Similar meaning
    "Python is a programming language",  # Different meaning
]

print("Creating embeddings...\n")
embeddings = []
for text in texts:
    result = text_embedder.run(text=text)
    embedding = result['embedding']
    embeddings.append(embedding)
    print(f"Text: {text}")
    print(f"Embedding shape: {np.array(embedding).shape}")
    print(f"First 5 values: {embedding[:5]}\n")

In [ ]:
#CELL-NO: 4
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    Calculate cosine similarity between two vectors.
    Returns value between -1 (opposite) and 1 (identical).
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Calculate similarities
print("Cosine Similarities:\n")
print(f"Text 1 vs Text 2 (similar meaning): {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"Text 1 vs Text 3 (different meaning): {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"Text 2 vs Text 3 (different meaning): {cosine_similarity(embeddings[1], embeddings[2]):.4f}")

print("\n💡 Higher similarity score = more semantically similar!")

### 3.2 Prepare Dataset for Semantic Search

Let's create a larger dataset to demonstrate semantic search.

In [ ]:
#CELL-NO: 5
def generate_ml_wikipedia_articles() -> List[Document]:
    """
    Generate sample Wikipedia-style articles about machine learning topics.
    In a real scenario, you'd load these from a dataset.
    """
    articles = [
        {
            "title": "Neural Networks",
            "content": """Neural networks are computing systems inspired by biological neural networks. 
            They consist of interconnected nodes (neurons) organized in layers. Information flows through 
            the network, with each connection having an associated weight. Training adjusts these weights 
            to minimize prediction error. Common architectures include feedforward networks, convolutional 
            neural networks (CNNs), and recurrent neural networks (RNNs).""",
            "category": "Deep Learning"
        },
        {
            "title": "Random Forest",
            "content": """Random Forest is an ensemble learning method that constructs multiple decision 
            trees during training. For classification, it outputs the mode of the classes from individual 
            trees. For regression, it outputs the mean prediction. The algorithm introduces randomness in 
            two ways: bootstrap sampling of training data and random feature selection at each split. 
            This reduces overfitting compared to single decision trees.""",
            "category": "Machine Learning"
        },
        {
            "title": "Gradient Descent",
            "content": """Gradient descent is an optimization algorithm used to minimize loss functions. 
            It iteratively moves in the direction of steepest descent, defined by the negative gradient. 
            The learning rate controls step size. Variants include batch gradient descent (uses all data), 
            stochastic gradient descent (uses one sample), and mini-batch gradient descent (uses small batches). 
            Adam and RMSprop are advanced adaptive learning rate methods.""",
            "category": "Optimization"
        },
        {
            "title": "Support Vector Machines",
            "content": """Support Vector Machines (SVM) are supervised learning models for classification 
            and regression. SVMs find the optimal hyperplane that maximizes the margin between classes. 
            Support vectors are data points closest to the hyperplane. The kernel trick allows SVMs to 
            handle non-linear decision boundaries by implicitly mapping data to higher dimensions. 
            Common kernels include linear, polynomial, and radial basis function (RBF).""",
            "category": "Machine Learning"
        },
        {
            "title": "Convolutional Neural Networks",
            "content": """Convolutional Neural Networks (CNNs) are deep learning models designed for 
            processing grid-like data such as images. Key components include convolutional layers (apply 
            filters to detect features), pooling layers (downsample to reduce dimensions), and fully 
            connected layers (perform classification). CNNs automatically learn hierarchical features, 
            from edges to complex patterns. Popular architectures include LeNet, AlexNet, VGG, and ResNet.""",
            "category": "Deep Learning"
        },
        {
            "title": "K-Means Clustering",
            "content": """K-Means is an unsupervised learning algorithm that partitions data into K clusters. 
            The algorithm iteratively assigns points to nearest centroid and updates centroids as cluster means. 
            Initialization methods like K-Means++ improve convergence. The elbow method helps determine optimal K. 
            K-Means assumes spherical clusters of similar size and struggles with irregular shapes. 
            Alternatives include DBSCAN and hierarchical clustering.""",
            "category": "Unsupervised Learning"
        },
        {
            "title": "Transformer Architecture",
            "content": """Transformers revolutionized NLP through self-attention mechanisms. Unlike RNNs, 
            they process entire sequences in parallel. Key components include multi-head attention (captures 
            different relationships), positional encoding (preserves sequence order), and feedforward networks. 
            The encoder-decoder structure enables seq2seq tasks. BERT uses encoder-only, GPT uses decoder-only. 
            Transformers scale effectively and form the basis of large language models.""",
            "category": "Deep Learning"
        },
        {
            "title": "Principal Component Analysis",
            "content": """Principal Component Analysis (PCA) is a dimensionality reduction technique that 
            identifies principal components - orthogonal directions of maximum variance. PCA projects data 
            onto lower-dimensional space while preserving most variance. It's computed via eigenvalue 
            decomposition of covariance matrix. Applications include visualization, noise reduction, and 
            feature extraction. PCA assumes linear relationships; alternatives include t-SNE and UMAP.""",
            "category": "Dimensionality Reduction"
        },
        {
            "title": "Reinforcement Learning",
            "content": """Reinforcement Learning (RL) trains agents to make decisions through trial and error. 
            An agent interacts with environment, receiving rewards for actions. The goal is to learn a policy 
            maximizing cumulative reward. Q-Learning learns action values, policy gradient methods directly 
            optimize policy. Deep RL combines neural networks with RL, enabling complex tasks like game playing. 
            Notable algorithms include DQN, A3C, and PPO.""",
            "category": "Reinforcement Learning"
        },
        {
            "title": "Decision Trees",
            "content": """Decision trees are supervised learning models that split data based on feature values. 
            Each internal node represents a test, branches represent outcomes, and leaves represent predictions. 
            Common splitting criteria include Gini impurity and information gain (entropy reduction). 
            Trees are interpretable and handle non-linear relationships but prone to overfitting. 
            Pruning techniques and ensemble methods like Random Forest address this limitation.""",
            "category": "Machine Learning"
        },
        {
            "title": "Recurrent Neural Networks",
            "content": """Recurrent Neural Networks (RNNs) process sequential data by maintaining hidden state. 
            Information from previous timesteps influences current predictions through recurrent connections. 
            Vanilla RNNs suffer from vanishing gradients in long sequences. LSTM (Long Short-Term Memory) 
            and GRU (Gated Recurrent Unit) use gating mechanisms to remember long-term dependencies. 
            Applications include language modeling, machine translation, and time series prediction.""",
            "category": "Deep Learning"
        },
        {
            "title": "Logistic Regression",
            "content": """Logistic regression is a statistical model for binary classification. Despite its name, 
            it's a classification algorithm. It models probability using the sigmoid function, producing outputs 
            between 0 and 1. Training maximizes likelihood via gradient descent. Multinomial logistic regression 
            extends to multiple classes. Regularization (L1/L2) prevents overfitting. Logistic regression is 
            interpretable, fast, and serves as baseline for classification tasks.""",
            "category": "Machine Learning"
        }
    ]
    
    return [
        Document(
            content=article["content"],
            meta={
                "title": article["title"],
                "category": article["category"],
                "source": "ml_wikipedia"
            }
        )
        for article in articles
    ]

# Generate articles
ml_articles = generate_ml_wikipedia_articles()
print(f"✅ Generated {len(ml_articles)} ML articles")
print("\nSample article:")
print(f"Title: {ml_articles[0].meta['title']}")
print(f"Category: {ml_articles[0].meta['category']}")
print(f"Content: {ml_articles[0].content[:150]}...")

### 3.3 Build Semantic Search Pipeline

In [ ]:
#CELL-NO: 6
# Initialize document store
semantic_store = InMemoryDocumentStore()

# Initialize document embedder
print("Initializing document embedder...")
doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

# Embed documents
print("\nEmbedding documents (this may take a moment)...")
embedded_docs = doc_embedder.run(documents=ml_articles)

# Write to store
semantic_store.write_documents(embedded_docs['documents'])

print(f"✅ Embedded and stored {semantic_store.count_documents()} documents")
print(f"\nEmbedding dimension: {len(embedded_docs['documents'][0].embedding)} dimensions")

In [ ]:
#CELL-NO: 7
# Build semantic search pipeline
semantic_pipeline = Pipeline()

# Add components
semantic_pipeline.add_component(
    "text_embedder", 
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
semantic_pipeline.add_component(
    "retriever", 
    InMemoryEmbeddingRetriever(document_store=semantic_store)
)

# Connect components
semantic_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

print("✅ Semantic search pipeline created")
print("\nPipeline structure:")
print(semantic_pipeline)

### 3.4 Compare BM25 vs Semantic Search

In [ ]:
#CELL-NO: 8
# Also create BM25 pipeline for comparison
bm25_store = InMemoryDocumentStore()
bm25_store.write_documents(ml_articles)

bm25_pipeline = Pipeline()
bm25_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=bm25_store))

print("✅ BM25 pipeline created for comparison")

In [ ]:
#CELL-NO: 9
def compare_retrieval_methods(query: str, top_k: int = 3):
    """
    Compare BM25 and semantic search side-by-side.
    """
    print("\n" + "="*100)
    print(f"Query: '{query}'")
    print("="*100)
    
    # BM25 results
    bm25_results = bm25_pipeline.run({
        "retriever": {"query": query, "top_k": top_k}
    })
    
    # Semantic results
    semantic_results = semantic_pipeline.run({
        "text_embedder": {"text": query},
        "retriever": {"top_k": top_k}
    })
    
    # Display side by side
    print("\n" + "-"*50 + " BM25 (Keyword) " + "-"*50)
    for i, doc in enumerate(bm25_results['retriever']['documents'], 1):
        print(f"\n{i}. {doc.meta['title']} (score: {doc.score:.4f})")
        print(f"   Category: {doc.meta['category']}")
        print(f"   Content: {doc.content[:120]}...")
    
    print("\n" + "-"*50 + " Semantic (Embeddings) " + "-"*50)
    for i, doc in enumerate(semantic_results['retriever']['documents'], 1):
        print(f"\n{i}. {doc.meta['title']} (score: {doc.score:.4f})")
        print(f"   Category: {doc.meta['category']}")
        print(f"   Content: {doc.content[:120]}...")

# Test with queries where semantic search excels
test_queries = [
    "How do artificial brains learn?",  # Semantic: Neural Networks
    "Organizing data into groups without labels",  # Semantic: K-Means
    "Image recognition deep learning",  # Semantic: CNNs
]

for query in test_queries:
    compare_retrieval_methods(query)

### Key Observations

- **BM25** excels when query contains exact keywords from documents
- **Semantic search** understands meaning and finds relevant docs even with different words
- **Semantic search** handles paraphrasing and vocabulary mismatch better

## 4. Hybrid Retrieval: Best of Both Worlds

Hybrid retrieval combines BM25 and semantic search for optimal results.

### 4.1 Build Hybrid Retrieval Pipeline

In [ ]:
#CELL-NO: 10
# Create a unified document store with embeddings
hybrid_store = InMemoryDocumentStore()
hybrid_store.write_documents(embedded_docs['documents'])

# Build hybrid pipeline
hybrid_pipeline = Pipeline()

# Add retrievers
hybrid_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
hybrid_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=hybrid_store)
)
hybrid_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=hybrid_store)
)

# Add document joiner to combine results
hybrid_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion")  # Smart merging strategy
)

# Connect components
hybrid_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
hybrid_pipeline.connect("bm25_retriever", "joiner")
hybrid_pipeline.connect("embedding_retriever", "joiner")

print("✅ Hybrid retrieval pipeline created")
print("\nPipeline structure:")
print(hybrid_pipeline)

### 4.2 Understanding Join Modes

The `DocumentJoiner` can combine results in different ways:

1. **concatenate**: Simply merge lists (duplicates possible)
2. **merge**: Combine and deduplicate (average scores)
3. **reciprocal_rank_fusion (RRF)**: Smart ranking fusion (recommended)

**RRF Formula:** `score = Σ(1 / (k + rank_i))` where k=60 typically

In [ ]:
#CELL-NO: 11
def test_hybrid_retrieval(query: str, top_k: int = 5):
    """
    Test hybrid retrieval and show results from both retrievers.
    """
    print("\n" + "="*100)
    print(f"Query: '{query}'")
    print("="*100)
    
    result = hybrid_pipeline.run({
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query, "top_k": top_k},
        "embedding_retriever": {"top_k": top_k}
    })
    
    print("\n🔍 Hybrid Retrieval Results (Reciprocal Rank Fusion):\n")
    for i, doc in enumerate(result['joiner']['documents'][:top_k], 1):
        print(f"{i}. {doc.meta['title']} (RRF score: {doc.score:.4f})")
        print(f"   Category: {doc.meta['category']}")
        print(f"   {doc.content[:100]}...\n")

# Test hybrid retrieval
test_queries = [
    "neural network architecture for images",
    "tree-based ensemble methods",
    "optimization algorithms for training models"
]

for query in test_queries:
    test_hybrid_retrieval(query)

### 4.3 Compare All Three Approaches

In [ ]:
#CELL-NO: 12
def comprehensive_comparison(query: str, top_k: int = 3):
    """
    Compare BM25, Semantic, and Hybrid retrieval.
    """
    print("\n" + "="*100)
    print(f"COMPREHENSIVE COMPARISON: '{query}'")
    print("="*100)
    
    # Get results from all three methods
    bm25_result = bm25_pipeline.run({"retriever": {"query": query, "top_k": top_k}})
    semantic_result = semantic_pipeline.run({
        "text_embedder": {"text": query},
        "retriever": {"top_k": top_k}
    })
    hybrid_result = hybrid_pipeline.run({
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query, "top_k": top_k},
        "embedding_retriever": {"top_k": top_k}
    })
    
    # Create comparison DataFrame
    methods = ['BM25', 'Semantic', 'Hybrid']
    results = [
        bm25_result['retriever']['documents'],
        semantic_result['retriever']['documents'],
        hybrid_result['joiner']['documents'][:top_k]
    ]
    
    data = []
    for method, docs in zip(methods, results):
        for rank, doc in enumerate(docs, 1):
            data.append({
                'Method': method,
                'Rank': rank,
                'Title': doc.meta['title'],
                'Score': f"{doc.score:.4f}"
            })
    
    df = pd.DataFrame(data)
    
    # Display as pivot table
    for method in methods:
        print(f"\n{method}:")
        method_df = df[df['Method'] == method][['Rank', 'Title', 'Score']]
        print(method_df.to_string(index=False))

# Test with different query types
comprehensive_comparison("supervised classification algorithm")
comprehensive_comparison("attention mechanism in NLP")

## 5. Query Expansion Techniques

Query expansion improves recall by generating additional queries or reformulations.

In [ ]:
#CELL-NO: 13
def expand_query_with_llm(original_query: str, num_variations: int = 3) -> List[str]:
    """
    Use an LLM to generate query variations.
    """
    prompt = f"""Generate {num_variations} alternative phrasings of this search query.
Each variation should ask for the same information but use different words.

Original query: {original_query}

Provide only the alternative queries, one per line, without numbering or explanations.
"""
    
    llm = OpenAIGenerator(
        api_key=Secret.from_env_var("OPENAI_API_KEY"),
        model="gpt-3.5-turbo",
        generation_kwargs={"temperature": 0.7}
    )
    
    result = llm.run(prompt=prompt)
    variations = result['replies'][0].strip().split('\n')
    
    # Clean up variations
    variations = [v.strip() for v in variations if v.strip()]
    
    return [original_query] + variations[:num_variations]

# Test query expansion
original = "How do neural networks learn?"
expanded = expand_query_with_llm(original, num_variations=3)

print("Query Expansion Example:\n")
print(f"Original: {original}\n")
print("Expanded queries:")
for i, query in enumerate(expanded[1:], 1):
    print(f"{i}. {query}")

In [ ]:
#CELL-NO: 14
def retrieve_with_query_expansion(query: str, top_k: int = 5) -> List[Document]:
    """
    Perform retrieval with query expansion.
    Retrieves results for multiple query variations and combines them.
    """
    # Expand query
    expanded_queries = expand_query_with_llm(query, num_variations=2)
    
    print(f"\nExpanded queries:")
    for i, q in enumerate(expanded_queries, 1):
        print(f"  {i}. {q}")
    
    # Retrieve for each query
    all_docs = []
    doc_scores = defaultdict(float)
    
    for exp_query in expanded_queries:
        result = hybrid_pipeline.run({
            "text_embedder": {"text": exp_query},
            "bm25_retriever": {"query": exp_query, "top_k": top_k},
            "embedding_retriever": {"top_k": top_k}
        })
        
        for doc in result['joiner']['documents']:
            # Aggregate scores for same document
            doc_scores[doc.id] = max(doc_scores[doc.id], doc.score)
            if doc.id not in [d.id for d in all_docs]:
                all_docs.append(doc)
    
    # Sort by aggregated scores
    all_docs.sort(key=lambda d: doc_scores[d.id], reverse=True)
    
    return all_docs[:top_k]

# Test query expansion retrieval
print("\n" + "="*100)
print("RETRIEVAL WITH QUERY EXPANSION")
print("="*100)

query = "clustering algorithm"
results = retrieve_with_query_expansion(query, top_k=5)

print("\n📊 Final Results:\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.meta['title']}")
    print(f"   {doc.content[:100]}...\n")

## 6. Reranking Strategies

Reranking improves precision by re-scoring retrieved documents with more sophisticated models.

### 6.1 Cross-Encoder Reranking

Cross-encoders score query-document pairs more accurately than bi-encoders.

In [ ]:
!uv add "transformers[torch,sentencepiece]"

In [ ]:
#CELL-NO: 15
# Build pipeline with reranking
rerank_pipeline = Pipeline()

# Add retrieval components
rerank_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
rerank_pipeline.add_component(
    "retriever",
    InMemoryEmbeddingRetriever(document_store=semantic_store)
)

# Add reranker (cross-encoder model)
rerank_pipeline.add_component(
    "ranker",
    TransformersSimilarityRanker(model="cross-encoder/ms-marco-MiniLM-L-6-v2")
)

# Connect components
rerank_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
rerank_pipeline.connect("retriever", "ranker")

print("✅ Pipeline with reranking created")

In [ ]:
#CELL-NO: 16
def compare_with_without_reranking(query: str, top_k_retrieve: int = 10, top_k_final: int = 5):
    """
    Compare retrieval with and without reranking.
    """
    print("\n" + "="*100)
    print(f"Query: '{query}'")
    print("="*100)
    
    # Without reranking
    without_rerank = semantic_pipeline.run({
        "text_embedder": {"text": query},
        "retriever": {"top_k": top_k_final}
    })
    
    # With reranking (retrieve more, then rerank to top_k)
    with_rerank = rerank_pipeline.run({
        "text_embedder": {"text": query},
        "retriever": {"top_k": top_k_retrieve},
        "ranker": {"query": query, "top_k": top_k_final}
    })
    
    # Display comparison
    print("\n" + "-"*45 + " WITHOUT Reranking " + "-"*45)
    for i, doc in enumerate(without_rerank['retriever']['documents'], 1):
        print(f"{i}. {doc.meta['title']} (score: {doc.score:.4f})")
    
    print("\n" + "-"*45 + " WITH Reranking " + "-"*45)
    for i, doc in enumerate(with_rerank['ranker']['documents'], 1):
        print(f"{i}. {doc.meta['title']} (score: {doc.score:.4f})")

# Test reranking
compare_with_without_reranking("deep learning for computer vision")
compare_with_without_reranking("unsupervised learning techniques")

### 6.2 Lost in the Middle Reranking

Research shows LLMs pay less attention to middle documents. This ranker addresses that.

In [ ]:
#CELL-NO: 17
# Create pipeline with LostInTheMiddle ranker
litm_pipeline = Pipeline()

litm_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
litm_pipeline.add_component(
    "retriever",
    InMemoryEmbeddingRetriever(document_store=semantic_store)
)
litm_pipeline.add_component(
    "ranker",
    LostInTheMiddleRanker(word_count_threshold=1024)  # Reorder long contexts
)

litm_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
litm_pipeline.connect("retriever", "ranker")

print("✅ Lost in the Middle ranker pipeline created")
print("\nThis ranker reorders documents to place most relevant ones at beginning and end,")
print("avoiding the 'lost in the middle' problem where LLMs miss mid-context information.")

## 7. Comprehensive Evaluation

Let's implement proper retrieval evaluation metrics.

In [ ]:
#CELL-NO: 18
def create_evaluation_dataset() -> List[Dict[str, Any]]:
    """
    Create evaluation dataset with queries and relevant document titles.
    """
    return [
        {
            "query": "How do artificial neural networks process information?",
            "relevant_docs": ["Neural Networks", "Convolutional Neural Networks", "Recurrent Neural Networks"]
        },
        {
            "query": "What is an ensemble method that uses multiple decision trees?",
            "relevant_docs": ["Random Forest", "Decision Trees"]
        },
        {
            "query": "Optimization algorithm for training machine learning models",
            "relevant_docs": ["Gradient Descent"]
        },
        {
            "query": "Dimensionality reduction technique for visualization",
            "relevant_docs": ["Principal Component Analysis"]
        },
        {
            "query": "Deep learning architecture for image recognition",
            "relevant_docs": ["Convolutional Neural Networks", "Neural Networks"]
        },
        {
            "query": "Unsupervised learning for grouping similar data",
            "relevant_docs": ["K-Means Clustering"]
        },
        {
            "query": "Model for binary classification problems",
            "relevant_docs": ["Logistic Regression", "Support Vector Machines", "Decision Trees"]
        },
        {
            "query": "Learning through rewards and penalties",
            "relevant_docs": ["Reinforcement Learning"]
        }
    ]

eval_dataset = create_evaluation_dataset()
print(f"✅ Created evaluation dataset with {len(eval_dataset)} queries")
print("\nSample evaluation case:")
print(f"Query: {eval_dataset[0]['query']}")
print(f"Relevant docs: {eval_dataset[0]['relevant_docs']}")

In [ ]:
#CELL-NO: 19
def calculate_recall_at_k(retrieved_titles: List[str], relevant_titles: List[str], k: int) -> float:
    """
    Calculate Recall@K: fraction of relevant documents retrieved in top K.
    """
    retrieved_set = set(retrieved_titles[:k])
    relevant_set = set(relevant_titles)
    
    if not relevant_set:
        return 0.0
    
    relevant_retrieved = retrieved_set.intersection(relevant_set)
    return len(relevant_retrieved) / len(relevant_set)

def calculate_precision_at_k(retrieved_titles: List[str], relevant_titles: List[str], k: int) -> float:
    """
    Calculate Precision@K: fraction of retrieved documents that are relevant.
    """
    retrieved_set = set(retrieved_titles[:k])
    relevant_set = set(relevant_titles)
    
    if not retrieved_set:
        return 0.0
    
    relevant_retrieved = retrieved_set.intersection(relevant_set)
    return len(relevant_retrieved) / len(retrieved_set)

def calculate_mrr(retrieved_titles: List[str], relevant_titles: List[str]) -> float:
    """
    Calculate Mean Reciprocal Rank: 1/rank of first relevant document.
    """
    relevant_set = set(relevant_titles)
    
    for i, title in enumerate(retrieved_titles, 1):
        if title in relevant_set:
            return 1.0 / i
    
    return 0.0

def calculate_ndcg_at_k(retrieved_titles: List[str], relevant_titles: List[str], k: int) -> float:
    """
    Calculate Normalized Discounted Cumulative Gain@K.
    """
    relevant_set = set(relevant_titles)
    
    # DCG: sum of (relevance / log2(position + 1))
    dcg = 0.0
    for i, title in enumerate(retrieved_titles[:k], 1):
        relevance = 1.0 if title in relevant_set else 0.0
        dcg += relevance / np.log2(i + 1)
    
    # IDCG: best possible DCG
    idcg = 0.0
    for i in range(min(len(relevant_titles), k)):
        idcg += 1.0 / np.log2(i + 2)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg

print("✅ Evaluation metrics implemented:")
print("  • Recall@K")
print("  • Precision@K")
print("  • Mean Reciprocal Rank (MRR)")
print("  • Normalized Discounted Cumulative Gain (NDCG@K)")

In [ ]:
#CELL-NO: 20
def evaluate_retrieval_system(pipeline, dataset: List[Dict], k_values: List[int] = [1, 3, 5]):
    """
    Comprehensive evaluation of a retrieval system.
    """
    results = {
        'recall': {k: [] for k in k_values},
        'precision': {k: [] for k in k_values},
        'ndcg': {k: [] for k in k_values},
        'mrr': []
    }
    
    print(f"\nEvaluating on {len(dataset)} queries...")
    
    for eval_case in tqdm(dataset):
        query = eval_case['query']
        relevant_titles = eval_case['relevant_docs']
        
        # Get retrieval results (adjust based on pipeline type)
        try:
            # Try hybrid pipeline format
            result = pipeline.run({
                "text_embedder": {"text": query},
                "bm25_retriever": {"query": query, "top_k": max(k_values)},
                "embedding_retriever": {"top_k": max(k_values)}
            })
            retrieved_docs = result['joiner']['documents']
        except:
            try:
                # Try semantic pipeline format
                result = pipeline.run({
                    "text_embedder": {"text": query},
                    "retriever": {"top_k": max(k_values)}
                })
                retrieved_docs = result['retriever']['documents']
            except:
                # Try BM25 pipeline format
                result = pipeline.run({
                    "retriever": {"query": query, "top_k": max(k_values)}
                })
                retrieved_docs = result['retriever']['documents']
        
        retrieved_titles = [doc.meta['title'] for doc in retrieved_docs]
        
        # Calculate metrics
        for k in k_values:
            results['recall'][k].append(calculate_recall_at_k(retrieved_titles, relevant_titles, k))
            results['precision'][k].append(calculate_precision_at_k(retrieved_titles, relevant_titles, k))
            results['ndcg'][k].append(calculate_ndcg_at_k(retrieved_titles, relevant_titles, k))
        
        results['mrr'].append(calculate_mrr(retrieved_titles, relevant_titles))
    
    # Calculate averages
    avg_results = {
        'recall': {k: np.mean(v) for k, v in results['recall'].items()},
        'precision': {k: np.mean(v) for k, v in results['precision'].items()},
        'ndcg': {k: np.mean(v) for k, v in results['ndcg'].items()},
        'mrr': np.mean(results['mrr'])
    }
    
    return avg_results

# Evaluate all three methods
print("\n" + "="*100)
print("COMPREHENSIVE RETRIEVAL EVALUATION")
print("="*100)

print("\n📊 Evaluating BM25...")
bm25_metrics = evaluate_retrieval_system(bm25_pipeline, eval_dataset)

print("\n📊 Evaluating Semantic Search...")
semantic_metrics = evaluate_retrieval_system(semantic_pipeline, eval_dataset)

print("\n📊 Evaluating Hybrid Retrieval...")
hybrid_metrics = evaluate_retrieval_system(hybrid_pipeline, eval_dataset)

In [ ]:
#CELL-NO: 21
# Display results in a nice table
def display_metrics(bm25_metrics, semantic_metrics, hybrid_metrics):
    """
    Display evaluation metrics in a formatted table.
    """
    print("\n" + "="*100)
    print("EVALUATION RESULTS")
    print("="*100)
    
    # Create comparison DataFrame
    data = []
    
    for k in [1, 3, 5]:
        data.append({
            'Metric': f'Recall@{k}',
            'BM25': f"{bm25_metrics['recall'][k]:.3f}",
            'Semantic': f"{semantic_metrics['recall'][k]:.3f}",
            'Hybrid': f"{hybrid_metrics['recall'][k]:.3f}"
        })
        data.append({
            'Metric': f'Precision@{k}',
            'BM25': f"{bm25_metrics['precision'][k]:.3f}",
            'Semantic': f"{semantic_metrics['precision'][k]:.3f}",
            'Hybrid': f"{hybrid_metrics['precision'][k]:.3f}"
        })
        data.append({
            'Metric': f'NDCG@{k}',
            'BM25': f"{bm25_metrics['ndcg'][k]:.3f}",
            'Semantic': f"{semantic_metrics['ndcg'][k]:.3f}",
            'Hybrid': f"{hybrid_metrics['ndcg'][k]:.3f}"
        })
    
    data.append({
        'Metric': 'MRR',
        'BM25': f"{bm25_metrics['mrr']:.3f}",
        'Semantic': f"{semantic_metrics['mrr']:.3f}",
        'Hybrid': f"{hybrid_metrics['mrr']:.3f}"
    })
    
    df = pd.DataFrame(data)
    print("\n", df.to_string(index=False))
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    metrics_to_plot = ['recall', 'precision', 'ndcg']
    k_values = [1, 3, 5]
    
    for idx, metric_name in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        bm25_values = [bm25_metrics[metric_name][k] for k in k_values]
        semantic_values = [semantic_metrics[metric_name][k] for k in k_values]
        hybrid_values = [hybrid_metrics[metric_name][k] for k in k_values]
        
        x = np.arange(len(k_values))
        width = 0.25
        
        ax.bar(x - width, bm25_values, width, label='BM25', alpha=0.8)
        ax.bar(x, semantic_values, width, label='Semantic', alpha=0.8)
        ax.bar(x + width, hybrid_values, width, label='Hybrid', alpha=0.8)
        
        ax.set_ylabel('Score')
        ax.set_title(f'{metric_name.upper()}@K')
        ax.set_xticks(x)
        ax.set_xticklabels([f'K={k}' for k in k_values])
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    # MRR comparison
    ax = axes[1, 1]
    methods = ['BM25', 'Semantic', 'Hybrid']
    mrr_values = [bm25_metrics['mrr'], semantic_metrics['mrr'], hybrid_metrics['mrr']]
    
    ax.bar(methods, mrr_values, alpha=0.8, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_ylabel('Score')
    ax.set_title('Mean Reciprocal Rank (MRR)')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('retrieval_evaluation.png', dpi=150, bbox_inches='tight')
    print("\n📊 Visualization saved to 'retrieval_evaluation.png'")
    plt.show()

display_metrics(bm25_metrics, semantic_metrics, hybrid_metrics)

## 8. Performance Optimization

Let's measure and optimize retrieval performance.

In [ ]:
#CELL-NO: 22
def benchmark_retrieval_latency(pipeline, queries: List[str], num_runs: int = 3) -> Dict[str, float]:
    """
    Measure retrieval latency.
    """
    latencies = []
    
    for query in queries:
        query_times = []
        for _ in range(num_runs):
            start = time.time()
            try:
                # Try different pipeline formats
                try:
                    pipeline.run({
                        "text_embedder": {"text": query},
                        "bm25_retriever": {"query": query, "top_k": 5},
                        "embedding_retriever": {"top_k": 5}
                    })
                except:
                    try:
                        pipeline.run({
                            "text_embedder": {"text": query},
                            "retriever": {"top_k": 5}
                        })
                    except:
                        pipeline.run({"retriever": {"query": query, "top_k": 5}})
            except Exception as e:
                print(f"Error: {e}")
                continue
            
            end = time.time()
            query_times.append((end - start) * 1000)  # Convert to ms
        
        if query_times:
            latencies.extend(query_times)
    
    return {
        'mean': np.mean(latencies),
        'median': np.median(latencies),
        'p95': np.percentile(latencies, 95),
        'p99': np.percentile(latencies, 99),
        'min': np.min(latencies),
        'max': np.max(latencies)
    }

# Benchmark all methods
test_queries = [
    "neural networks",
    "machine learning algorithms",
    "deep learning architectures"
]

print("\n" + "="*100)
print("LATENCY BENCHMARKS (milliseconds)")
print("="*100)

print("\nBenchmarking BM25...")
bm25_latency = benchmark_retrieval_latency(bm25_pipeline, test_queries)

print("Benchmarking Semantic Search...")
semantic_latency = benchmark_retrieval_latency(semantic_pipeline, test_queries)

print("Benchmarking Hybrid Retrieval...")
hybrid_latency = benchmark_retrieval_latency(hybrid_pipeline, test_queries)

# Display results
latency_data = [
    {'Method': 'BM25', **{k: f"{v:.2f}" for k, v in bm25_latency.items()}},
    {'Method': 'Semantic', **{k: f"{v:.2f}" for k, v in semantic_latency.items()}},
    {'Method': 'Hybrid', **{k: f"{v:.2f}" for k, v in hybrid_latency.items()}}
]

latency_df = pd.DataFrame(latency_data)
print("\n", latency_df.to_string(index=False))

print("\n💡 Insights:")
print("  • BM25 is typically fastest (no embedding computation)")
print("  • Semantic search requires embedding the query")
print("  • Hybrid combines both, so it's slowest but most accurate")
print("  • For production, consider caching embeddings and using approximate search")

## 9. Choosing the Right Strategy

Decision guide for selecting retrieval strategies.

In [ ]:
#CELL-NO: 23
# Create decision guide
decision_guide = pd.DataFrame([
    {
        'Scenario': 'Exact keyword matching needed',
        'Recommended': 'BM25',
        'Reason': 'Fast and precise for keyword queries'
    },
    {
        'Scenario': 'Semantic understanding required',
        'Recommended': 'Semantic (Dense)',
        'Reason': 'Handles paraphrasing and synonyms'
    },
    {
        'Scenario': 'Best overall quality needed',
        'Recommended': 'Hybrid',
        'Reason': 'Combines strengths of both approaches'
    },
    {
        'Scenario': 'Limited compute budget',
        'Recommended': 'BM25',
        'Reason': 'Fastest, no embedding computation'
    },
    {
        'Scenario': 'Multilingual documents',
        'Recommended': 'Semantic (Dense)',
        'Reason': 'Cross-lingual embeddings available'
    },
    {
        'Scenario': 'High precision critical',
        'Recommended': 'Hybrid + Reranking',
        'Reason': 'Reranker refines initial retrieval'
    },
    {
        'Scenario': 'High recall needed',
        'Recommended': 'Semantic + Query Expansion',
        'Reason': 'Finds more relevant variations'
    },
    {
        'Scenario': 'Real-time latency requirements',
        'Recommended': 'BM25',
        'Reason': 'Lowest latency option'
    }
])

print("\n" + "="*100)
print("RETRIEVAL STRATEGY DECISION GUIDE")
print("="*100 + "\n")
print(decision_guide.to_string(index=False))

## 10. Exercises & Challenges

### Exercise 1: Implement Custom Scoring

Create a custom scoring function that combines BM25 and semantic scores with custom weights.

In [ ]:
#CELL-NO: 24
# Your code here
# TODO: Implement weighted combination of BM25 and semantic scores
# TODO: Experiment with different weight ratios (e.g., 0.3 BM25 + 0.7 semantic)
# TODO: Evaluate which weights work best for your use case

### Exercise 2: Add Domain-Specific Embeddings

Replace the general-purpose embedding model with a domain-specific one (e.g., biomedical, legal, code).

In [ ]:
#CELL-NO: 25
# Your code here
# TODO: Research domain-specific embedding models on Hugging Face
# TODO: Rebuild semantic pipeline with specialized model
# TODO: Compare performance with general model

### Exercise 3: Implement MMR (Maximal Marginal Relevance)

MMR promotes diversity in retrieved documents to avoid redundancy.

In [ ]:
#CELL-NO: 26
# Your code here
# TODO: Implement MMR algorithm
# MMR = λ * relevance - (1-λ) * max_similarity_to_selected
# TODO: Test with queries that might return similar documents

### Challenge: Build Multi-Stage Retrieval

Implement a cascading retrieval system:
1. Fast first-stage retrieval (BM25) gets top 100 candidates
2. Semantic reranking narrows to top 20
3. Cross-encoder reranking for final top 5

Measure latency vs. quality tradeoffs.

In [ ]:
#CELL-NO: 27
# Your code here
# TODO: Build cascading pipeline
# TODO: Benchmark each stage
# TODO: Compare with single-stage retrieval

## 11. Summary & Next Steps

### What You've Learned

✅ **Dense Embeddings**
- How embeddings capture semantic meaning
- Implementing semantic search with Haystack
- Cosine similarity for finding relevant documents

✅ **Hybrid Retrieval**
- Combining BM25 and semantic search
- Reciprocal Rank Fusion for merging results
- When hybrid outperforms single methods

✅ **Advanced Techniques**
- Query expansion with LLMs
- Cross-encoder reranking
- Lost in the Middle ranker

✅ **Comprehensive Evaluation**
- Recall@K, Precision@K, MRR, NDCG
- Systematic benchmarking
- Latency vs. quality tradeoffs

### Key Takeaways

1. **No one-size-fits-all**: Choose strategy based on your use case
2. **Hybrid often wins**: Combines keyword precision with semantic understanding
3. **Reranking helps**: Improves precision at cost of latency
4. **Evaluate quantitatively**: Use proper metrics, not just manual inspection
5. **Consider costs**: Balance quality, latency, and computational requirements

### What's Next?

In **Notebook 3 - Advanced RAG Patterns & Agentic Behavior**, you'll learn:

- 🤖 **Conditional routing** - Dynamic pipeline decisions
- 🌐 **Web search fallback** - When local knowledge isn't enough
- ✅ **Self-correcting RAG** - Validate and improve responses
- 💬 **Conversational RAG** - Multi-turn dialogue with memory
- 🎯 **Adaptive retrieval** - Route based on query complexity

### Additional Resources

- [Dense Passage Retrieval Paper](https://arxiv.org/abs/2004.04906)
- [Sentence Transformers Documentation](https://www.sbert.net/)
- [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [Lost in the Middle Paper](https://arxiv.org/abs/2307.03172)

---

**Great work!** You now have advanced retrieval capabilities. Continue to **Notebook 3** to make your RAG systems intelligent and adaptive! 🚀